# 한국어 감성분석 (NSMC + scikit-learn)

NSMC(Naver Sentiment Movie Corpus) 데이터셋을 활용한 한국어 감성분석 모델 구축 과정을 정리한 노트북입니다.

- **데이터**: NSMC 영화 리뷰 (학습 15만건, 테스트 5만건)
- **전처리**: Okt 형태소 분석 + 불용어/품사 필터링 + TF-IDF
- **모델**: MultinomialNB, LogisticRegression, LinearSVC, SGDClassifier 비교
- **3분류 전략**: 2분류 학습 후 예측 확률 임계값으로 중립 구분

## 1. 데이터 다운로드 및 로드

In [ ]:
import os
import sys
import urllib.request

import pandas as pd

# 프로젝트 루트를 path에 추가
sys.path.insert(0, os.path.abspath(".."))

DATA_DIR = os.path.join("..", "data")
BASE_URL = "https://raw.githubusercontent.com/e9t/nsmc/master"

for filename in ["ratings_train.txt", "ratings_test.txt"]:
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"다운로드 중: {filename}")
        urllib.request.urlretrieve(f"{BASE_URL}/{filename}", filepath)
    else:
        print(f"이미 존재: {filename}")

train_df = pd.read_csv(os.path.join(DATA_DIR, "ratings_train.txt"), sep="\t")
test_df = pd.read_csv(os.path.join(DATA_DIR, "ratings_test.txt"), sep="\t")

print(f"학습 데이터: {len(train_df):,}건")
print(f"테스트 데이터: {len(test_df):,}건")

## 2. 데이터 탐색 (EDA)

In [ ]:
train_df.head(10)

In [ ]:
train_df.info()
print(f"\n결측치:\n{train_df.isnull().sum()}")
print(f"\n중복 리뷰 수: {train_df['document'].duplicated().sum():,}건")

In [ ]:
print("라벨 분포:")
print(train_df["label"].value_counts())
print(f"\n긍정 비율: {train_df['label'].mean():.2%}")
print(f"부정 비율: {1 - train_df['label'].mean():.2%}")

In [ ]:
# 리뷰 길이 분포
train_df["doc_len"] = train_df["document"].astype(str).str.len()

print("리뷰 길이 통계:")
print(train_df["doc_len"].describe())
print(f"\n긍정 리뷰 평균 길이: {train_df[train_df['label']==1]['doc_len'].mean():.1f}자")
print(f"부정 리뷰 평균 길이: {train_df[train_df['label']==0]['doc_len'].mean():.1f}자")

## 3. 전처리

- 특수문자 제거 (한글, 영문, 숫자, 공백만 유지)
- Okt 형태소 분석기로 토큰화 (stem=True)
- 불필요한 품사 제외 (조사, 접미사, 구두점)
- 불용어 및 1글자 토큰 제거

In [ ]:
import re
from konlpy.tag import Okt

okt = Okt()

STOPWORDS = {
    "의", "가", "이", "은", "들", "는", "과",
    "도", "를", "으로", "자", "에", "와", "한", "에서",
    "로", "에게", "뿐", "의거", "것", "수", "거", "게",
    "그", "또", "다", "나", "너", "저",
    "우리", "그것", "이것", "저것", "여기", "거기", "저기",
    "때문", "위해", "통해", "대해", "관해",
}

EXCLUDE_POS = {"Josa", "Suffix", "Punctuation"}


def tokenize(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", "", text)
    tagged = okt.pos(text, stem=True)
    tokens = [
        word for word, pos in tagged
        if pos not in EXCLUDE_POS and word not in STOPWORDS and len(word) > 1
    ]
    return " ".join(tokens)


# 전처리 예시
sample_texts = ["이 영화 정말 재밌다!", "최악의 영화... 돈 아깝다", "그냥 그저 그래요"]
for text in sample_texts:
    print(f"{text}  →  {tokenize(text)}")

In [ ]:
from src.preprocess import load_data, preprocess_dataframe

# 결측치/중복 제거 후 토큰화 (시간이 걸립니다)
train_df = load_data(os.path.join(DATA_DIR, "ratings_train.txt"))
test_df = load_data(os.path.join(DATA_DIR, "ratings_test.txt"))

print(f"결측치/중복 제거 후 - 학습: {len(train_df):,}건, 테스트: {len(test_df):,}건")

train_df = preprocess_dataframe(train_df)
test_df = preprocess_dataframe(test_df)

print(f"전처리 후 - 학습: {len(train_df):,}건, 테스트: {len(test_df):,}건")

In [ ]:
# 전처리 결과 확인
train_df[["document", "tokens", "label"]].head(10)

## 4. TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text = train_df["tokens"]
y_train = train_df["label"]
X_test_text = test_df["tokens"]
y_test = test_df["label"]

tfidf = TfidfVectorizer(
    max_features=80000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=3,
    max_df=0.9,
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f"TF-IDF 행렬 크기: {X_train.shape}")
print(f"피처 수: {len(tfidf.get_feature_names_out()):,}개")

## 5. 모델 학습 및 비교

4개 분류기를 GridSearchCV로 하이퍼파라미터 탐색 후 비교합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

classifiers = {
    "MultinomialNB": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.01, 0.1, 0.5, 1.0]},
    },
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {"C": [0.5, 1.0, 2.0, 5.0]},
    },
    "LinearSVC": {
        "model": LinearSVC(max_iter=2000),
        "params": {"C": [0.5, 1.0, 2.0, 5.0]},
    },
    "SGDClassifier": {
        "model": SGDClassifier(loss="modified_huber", max_iter=1000),
        "params": {"alpha": [1e-4, 1e-5, 1e-6]},
    },
}

results = {}

for name, config in classifiers.items():
    print(f"\n{'='*50}")
    print(f"모델: {name}")
    print(f"{'='*50}")

    grid = GridSearchCV(
        config["model"], config["params"],
        cv=5, scoring="accuracy", n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    clf = grid.best_estimator_

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results[name] = {
        "best_params": grid.best_params_,
        "cv_score": grid.best_score_,
        "test_acc": acc,
        "model": clf,
    }

    print(f"최적 파라미터: {grid.best_params_}")
    print(f"교차 검증 정확도: {grid.best_score_:.4f}")
    print(f"테스트 정확도: {acc:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['부정', '긍정'])}")

## 6. 모델 비교 요약

In [ ]:
summary = pd.DataFrame({
    name: {
        "최적 파라미터": str(r["best_params"]),
        "교차 검증 정확도": f"{r['cv_score']:.4f}",
        "테스트 정확도": f"{r['test_acc']:.4f}",
    }
    for name, r in results.items()
}).T

summary

In [ ]:
# 최적 모델의 혼동 행렬
best_name = max(results, key=lambda k: results[k]["test_acc"])
best_clf = results[best_name]["model"]
y_pred_best = best_clf.predict(X_test)

print(f"최적 모델: {best_name} (정확도: {results[best_name]['test_acc']:.4f})")
print(f"\n혼동 행렬:")

cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(cm, index=["실제 부정", "실제 긍정"], columns=["예측 부정", "예측 긍정"])
cm_df

## 7. 모델 저장

In [ ]:
import joblib

MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "best_model.joblib")
tfidf_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.joblib")

joblib.dump(best_clf, model_path)
joblib.dump(tfidf, tfidf_path)

print(f"모델 저장: {model_path}")
print(f"벡터라이저 저장: {tfidf_path}")

## 8. 감성 예측 테스트 (3분류: 긍정/중립/부정)

2분류 모델의 예측 확률을 기반으로 중립을 구분합니다.
- 긍정 확률 > 0.6 → **긍정**
- 긍정 확률 0.4~0.6 → **중립**
- 긍정 확률 < 0.4 → **부정**

In [ ]:
NEUTRAL_LOW = 0.4
NEUTRAL_HIGH = 0.6


def predict_sentiment(text, model, vectorizer):
    tokens = tokenize(text)
    vec = vectorizer.transform([tokens])

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(vec)[0]
        pos_prob = proba[1]
        if NEUTRAL_LOW <= pos_prob <= NEUTRAL_HIGH:
            return "중립", pos_prob
        elif pos_prob > NEUTRAL_HIGH:
            return "긍정", pos_prob
        else:
            return "부정", 1 - pos_prob
    else:
        decision = model.decision_function(vec)[0]
        if -0.5 <= decision <= 0.5:
            return "중립", abs(decision)
        elif decision > 0.5:
            return "긍정", decision
        else:
            return "부정", abs(decision)


# 저장된 모델 로드
model = joblib.load(model_path)
tfidf_loaded = joblib.load(tfidf_path)

# 테스트 문장들
test_sentences = [
    "이 영화 정말 재밌다",
    "최악의 영화 두번 다시 안본다",
    "그냥 그저 그래요",
    "배우 연기는 좋은데 스토리가 별로",
    "인생 영화 감동적이었다",
    "시간 낭비였다",
    "볼만 했어요",
]

print(f"{'텍스트':<30} {'감성':<6} {'신뢰도'}")
print("-" * 50)
for text in test_sentences:
    label, confidence = predict_sentiment(text, model, tfidf_loaded)
    print(f"{text:<30} {label:<6} {confidence:.4f}")